# 04 - Robustness checks
The main result (notebook 02) tests only on 9 days of late September. Here we check whether the conclusions hold under other kinds of splits. These are checks only: no tuning, the same Random Forest settings every time.

1. **Blocked split inside every window:** all four seasons appear in training.
2. **Leave one window out:** train on three seasons, test on the fourth (a season the model has never seen).
3. **Unseen routes:** train on 12 routes, test on 4 routes the model has never seen.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from utils import HISTORY, TARGET, best_forest, load_data, make_pipeline, planning_features

sns.set_theme(style="whitegrid")
df = load_data()
planning = planning_features(df)
weather = ["temperature_2m", "apparent_temperature", "relative_humidity_2m", "dew_point_2m", "precipitation",
           "snowfall", "pressure_msl", "cloud_cover", "wind_speed_10m", "wind_gusts_10m"]
feature_sets = {
    "planning": planning,
    "planning minus weather": [c for c in planning if c not in weather],
    "history": planning + HISTORY,
}

## Helper functions
- `baseline` = the training average for the same route, hour and weekday/weekend (same baseline as notebook 02). For unseen routes we drop the route from the key.
- `compare` fits each feature set on the training part and returns the MAE on the test part.

In [2]:
def baseline(train_part, test_part, keys=("bus_route", "hour", "is_weekend")):
    keys = list(keys)
    typical = train_part.groupby(keys)[TARGET].mean().rename("typical").reset_index()
    merged = test_part[keys].merge(typical, on=keys, how="left")
    pred = merged["typical"].fillna(train_part[TARGET].mean()).to_numpy()
    return mean_absolute_error(test_part[TARGET], pred)

def compare(train_part, test_part, baseline_keys=("bus_route", "hour", "is_weekend")):
    result = {"baseline": baseline(train_part, test_part, baseline_keys)}
    for name, features in feature_sets.items():
        pipe = make_pipeline(best_forest(), features).fit(train_part[features], train_part[TARGET])
        result[name] = mean_absolute_error(test_part[TARGET], pipe.predict(test_part[features]))
    return result

## 1. Blocked split inside every window
In each 2-week window: first 10 days = train, last 4 days (Thursday to Sunday) = test. We use 4 days and not 2 because every window starts on a Monday, so the last 2 days would always be a weekend. The history features only look at the past, so no test values leak into the training features.

In [3]:
day_in_window = df.groupby("window_id")["transit_timestamp"].transform(lambda s: (s.dt.normalize() - s.dt.normalize().min()).dt.days)
blocked_train = df[day_in_window < 10]
blocked_test = df[day_in_window >= 10]
print(len(blocked_train), len(blocked_test))
blocked = pd.Series(compare(blocked_train, blocked_test), name="test MAE").round(1)
blocked

15360 6144


baseline                  63.3
planning                  47.3
planning minus weather    53.6
history                   39.3
Name: test MAE, dtype: float64

**Reading the blocked split:** when every season is in the training data, the planning model scores MAE 47.3 against 63.3 for the baseline (about 25% better), which agrees with the main result (28%). Removing weather makes it worse (53.6), and the history model is best (39.3).

## 2. Leave one window out
Each window (Jan, Apr, Jul, Sep) is held out in turn. This is the hardest test of whether the model carries over to a season it has not seen.

In [4]:
rows = []
for w in sorted(df["window_id"].unique()):
    result = compare(df[df["window_id"] != w], df[df["window_id"] == w])
    rows.append({"held-out window": w, **result})
loo = pd.DataFrame(rows).set_index("held-out window")
loo.loc["mean"] = loo.mean()
loo.round(1)

,baseline,planning,planning minus weather,history
held-out window,,,,
0,87.7,87.0,85.0,49.1
1,75.5,68.1,70.0,48.4
2,73.4,60.9,71.1,54.8
3,78.9,74.5,81.6,46.1
mean,78.9,72.6,76.9,49.6


**Reading leave-one-window-out:** this is the honest warning. On a season the model has never seen, the planning model averages MAE 72.6 against 78.9 for the baseline, which is only **about 8% better**. For January it is no better at all (87.0 vs 87.7). The history model stays good (49.6), because recent ridership tells it the current level.

So the 28% improvement in notebook 02 holds when the model has seen similar seasons in training. With only four 2-week windows, the model has little experience of seasonal level changes. More windows (for example 8 instead of 4) would probably help the most, and is cheap to do with the dataset builder.

**Weather:** removing weather makes the planning model worse in 3 of 4 held-out windows (average 76.9 vs 72.6), and in the blocked split too (53.6 vs 47.3). So weather helps even when it cannot be acting as a season label. Still, do not read this as 'rain causes fewer riders': the weather columns follow a daily cycle (for example humidity falls in the afternoon), so they partly re-encode the time of day. We can say weather is a **useful input**, not that we proved a cause.

## 3. Unseen routes
`GroupKFold` keeps all rows of a route in the same fold, so the 4 test routes in each fold are completely new. Time is not split here, so this only tells us about new routes, not new dates. The baseline uses the hour and weekday/weekend average over all training routes.

In [5]:
rows = []
for fold, (tr, te) in enumerate(GroupKFold(n_splits=4).split(df, groups=df["bus_route"])):
    train_part, test_part = df.iloc[tr], df.iloc[te]
    result = compare(train_part, test_part, baseline_keys=("hour", "is_weekend"))
    rows.append({"fold": fold, "test routes": ", ".join(sorted(test_part["bus_route"].unique())),
                 "mean riders": test_part[TARGET].mean(), **result})
unseen = pd.DataFrame(rows).set_index("fold")
unseen.drop(columns="history").round(1)

,test routes,mean riders,baseline,planning,planning minus weather
fold,,,,,
0,"B6, BX9, M86+, Q65",487.5,119.4,125.6,128.5
1,"B41, BX36, M4, Q58",476.3,142.5,126.8,128.0
2,"B35, BX19, M15+, Q27",531.7,162.8,149.7,150.7
3,"B1, BX12+, M101, Q25",507.5,109.1,129.7,127.9


**Reading unseen routes:** the planning model averages MAE 133 against 133 for the baseline, so it is **no better than the average hour profile** on a route it has never seen. Errors are around 27% of the average ridership. The model learns each route's size from the route name and its GTFS facts, and cannot guess the size of a new route from 12 others.

**What this means for the app:** it must only offer the 16 routes it was trained on. Predicting a new route would need more routes in training (or route features such as population nearby).